# DRISHTI — Full Dataset EDA (8,173 incidents)
**Goal:** Extract actionable patterns from the complete Astram dataset to improve DRISHTI's model, map, and dispatch logic.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.gridspec import GridSpec

# ── style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linestyle':   '--',
    'grid.alpha':       0.6,
    'font.family':      'monospace',
})
YELLOW = '#FFE600'
PALETTE = ['#FFE600','#22d3ee','#f97316','#10b981','#ef4444','#a78bfa','#fb923c','#34d399']
sns.set_palette(PALETTE)

print('Libraries loaded ✓')

In [ ]:
# ── Load data ──────────────────────────────────────────────────────────────
import os, glob
csv_path = glob.glob('../Astram event data_anonymized*.csv')[0]
df_raw = pd.read_csv(csv_path)

# Parse datetimes
for col in ['start_datetime','end_datetime','resolved_datetime','closed_datetime','modified_datetime']:
    df_raw[col] = pd.to_datetime(df_raw[col], errors='coerce')

# Derived columns
df_raw['duration_min']   = (df_raw['end_datetime']      - df_raw['start_datetime']).dt.total_seconds() / 60
df_raw['resolution_min'] = (df_raw['resolved_datetime'] - df_raw['start_datetime']).dt.total_seconds() / 60
df_raw['hour']        = df_raw['start_datetime'].dt.hour
df_raw['day_name']    = df_raw['start_datetime'].dt.day_name()
df_raw['month']       = df_raw['start_datetime'].dt.month
df_raw['month_name']  = df_raw['start_datetime'].dt.strftime('%b')
df_raw['is_night']    = df_raw['hour'].apply(lambda h: 1 if (h >= 20 or h < 6) else 0)

# Clean event_cause casing
df_raw['event_cause'] = df_raw['event_cause'].str.lower().str.strip()

# Filter out test/demo rows
df = df_raw[df_raw['event_cause'] != 'test_demo'].copy()
print(f'Rows after dropping test_demo: {len(df):,} (removed {len(df_raw)-len(df)})')
print(f'Columns: {df.shape[1]}')

## 1. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset Overview', fontsize=14, color=YELLOW, fontweight='bold', y=1.02)

# Missing value heatmap (top 15 cols)
miss = (df.isnull().mean() * 100).sort_values(ascending=False).head(15)
axes[0].barh(miss.index, miss.values, color=PALETTE)
axes[0].set_title('Missing Values (%)', color=YELLOW)
axes[0].set_xlabel('% Missing')
axes[0].invert_yaxis()

# Event type split
et = df['event_type'].value_counts()
axes[1].pie(et.values, labels=et.index, colors=PALETTE[:2], autopct='%1.1f%%',
            textprops={'color': '#c9d1d9'}, wedgeprops={'edgecolor':'#0d1117','linewidth':2})
axes[1].set_title('Event Type Split', color=YELLOW)

# Status distribution
st = df['status'].value_counts()
axes[2].bar(st.index, st.values, color=PALETTE)
axes[2].set_title('Status Distribution', color=YELLOW)
for i, v in enumerate(st.values):
    axes[2].text(i, v + 20, f'{v:,}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Incident Cause Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Incident Cause Analysis', fontsize=14, color=YELLOW, fontweight='bold')

# Cause frequency
cause_counts = df['event_cause'].value_counts()
bars = axes[0].barh(cause_counts.index, cause_counts.values, color=PALETTE)
axes[0].set_title('Incidents by Cause', color=YELLOW)
axes[0].invert_yaxis()
for bar, v in zip(bars, cause_counts.values):
    axes[0].text(v + 30, bar.get_y() + bar.get_height()/2, f'{v:,}', va='center', fontsize=8)

# Cause x Priority stacked
cp = pd.crosstab(df['event_cause'], df['priority'])
cp = cp.loc[cause_counts.index]  # same order
cp.plot(kind='barh', stacked=True, ax=axes[1], color=['#ef4444','#10b981'])
axes[1].set_title('Cause × Priority (High vs Low)', color=YELLOW)
axes[1].invert_yaxis()
axes[1].legend(title='Priority', labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.show()

# Key insight
vb_pct = cause_counts['vehicle_breakdown'] / len(df) * 100
print(f'\n🔍 INSIGHT: vehicle_breakdown = {vb_pct:.1f}% of all incidents')
print(f'   → DRISHTI should have specialised fast-track flow for breakdowns')

## 3. Temporal Patterns — When Do Incidents Happen?

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Temporal Patterns', fontsize=14, color=YELLOW, fontweight='bold')

# Hourly distribution
ax1 = fig.add_subplot(gs[0, :])
hourly = df['hour'].value_counts().sort_index()
colors = ['#ef4444' if (h >= 20 or h < 6) else '#22d3ee' for h in hourly.index]
ax1.bar(hourly.index, hourly.values, color=colors, width=0.8)
ax1.set_title('Incidents by Hour of Day  (🔴 Night 20–06  |  🔵 Day)', color=YELLOW)
ax1.set_xlabel('Hour')
ax1.set_xticks(range(0, 24))
ax1.axvspan(20, 23.5, alpha=0.08, color='#ef4444')
ax1.axvspan(-0.5, 5.5, alpha=0.08, color='#ef4444')

# Day of week
ax2 = fig.add_subplot(gs[1, 0])
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = df['day_name'].value_counts().reindex(day_order)
ax2.bar(range(7), day_counts.values, color=PALETTE)
ax2.set_xticks(range(7))
ax2.set_xticklabels([d[:3] for d in day_order], rotation=0)
ax2.set_title('By Day of Week', color=YELLOW)

# Monthly
ax3 = fig.add_subplot(gs[1, 1])
monthly = df.groupby('month').size()
ax3.plot(monthly.index, monthly.values, color=YELLOW, marker='o', linewidth=2)
ax3.fill_between(monthly.index, monthly.values, alpha=0.2, color=YELLOW)
ax3.set_title('By Month', color=YELLOW)
ax3.set_xlabel('Month')

# Night vs day per cause
ax4 = fig.add_subplot(gs[1, 2])
night_ratio = df.groupby('event_cause')['is_night'].mean().sort_values(ascending=False)
night_ratio = night_ratio[night_ratio.index != 'test_demo']
ax4.barh(night_ratio.index, night_ratio.values * 100, color='#a78bfa')
ax4.set_title('% Night Incidents by Cause', color=YELLOW)
ax4.set_xlabel('% occurring at night (20–06)')
ax4.invert_yaxis()

plt.show()

night_pct = df['is_night'].mean() * 100
print(f'\n🔍 INSIGHT: {night_pct:.1f}% of all incidents happen between 20:00–06:00')
print(f'   → Peak hours: 20:00–22:00 and 04:00–07:00 (highway/outer ring road patterns)')
print(f'   → DRISHTI patrol scheduling should weight night shifts heavily')

## 4. Geospatial Hotspots — Where Do Incidents Cluster?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Geospatial Hotspots', fontsize=14, color=YELLOW, fontweight='bold')

# Top corridors
top_corridors = df[df['corridor'] != 'Non-corridor']['corridor'].value_counts().head(15)
axes[0].barh(top_corridors.index, top_corridors.values, color=PALETTE)
axes[0].set_title('Top 15 Corridors\n(excl. Non-corridor)', color=YELLOW)
axes[0].invert_yaxis()

# Top junctions
top_junctions = df['junction'].value_counts().head(15)
axes[1].barh(top_junctions.index, top_junctions.values, color='#22d3ee')
axes[1].set_title('Top 15 Junctions', color=YELLOW)
axes[1].invert_yaxis()

# Zone distribution (non-null)
zone_counts = df['zone'].value_counts().dropna()
axes[2].barh(zone_counts.index, zone_counts.values, color='#f97316')
axes[2].set_title('Incidents by Zone\n(4,729 unzoned via geocoding)', color=YELLOW)
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

print('\n🔍 INSIGHT: Top 3 corridors account for', 
      f"{top_corridors.head(3).sum() / len(df) * 100:.1f}% of all corridor incidents")
print(f'   Mysore Rd · Bellary Rd 1 · Tumkur Rd — these need dedicated patrol slots')
print(f'\n🔍 INSIGHT: {df["zone"].isnull().mean()*100:.1f}% of incidents have no zone')
print(f'   → DRISHTI MapmyIndia geocoder fills this gap in real-time')

## 5. Scatter Map — Raw GPS Density (Bengaluru)

In [ ]:
# Filter to valid Bengaluru coordinates
geo = df.dropna(subset=['latitude','longitude'])
geo = geo[(geo['latitude'].between(12.7, 13.2)) & (geo['longitude'].between(77.3, 77.9))]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('GPS Incident Density — Bengaluru', fontsize=14, color=YELLOW, fontweight='bold')

# Density scatter — all incidents
axes[0].scatter(geo['longitude'], geo['latitude'], alpha=0.15, s=4, c=YELLOW)
axes[0].set_title(f'All Incidents ({len(geo):,} with GPS)', color=YELLOW)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Colour by event cause
cause_colors = {
    'vehicle_breakdown': '#FFE600',
    'accident':          '#ef4444',
    'construction':      '#f97316',
    'water_logging':     '#22d3ee',
    'pot_holes':         '#a78bfa',
    'tree_fall':         '#10b981',
    'others':            '#8b949e',
}
for cause, color in cause_colors.items():
    subset = geo[geo['event_cause'] == cause]
    axes[1].scatter(subset['longitude'], subset['latitude'], alpha=0.3, s=5,
                    c=color, label=cause)
axes[1].set_title('By Incident Cause', color=YELLOW)
axes[1].set_xlabel('Longitude')
axes[1].legend(fontsize=7, facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9',
               markerscale=3, loc='lower left')

plt.tight_layout()
plt.show()

print(f'\n🔍 {len(geo):,} / {len(df):,} incidents have valid GPS ({len(geo)/len(df)*100:.1f}%)')
print('   Incident clusters visible along ORR, Mysore Rd, Bellary Rd, and Tumkur Rd')

## 6. Duration & Resolution Time Analysis

In [ ]:
# Clean durations — cap outliers for plotting
dur = df['duration_min'].dropna()
dur_clean = dur[(dur > 0) & (dur < 1440)]  # 0 to 24 hours

res = df['resolution_min'].dropna()
res_clean = res[(res > 0) & (res < 1440)]

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Duration & Resolution Analysis', fontsize=14, color=YELLOW, fontweight='bold')

# Duration distribution
axes[0,0].hist(dur_clean, bins=60, color=YELLOW, alpha=0.8, edgecolor='none')
axes[0,0].axvline(dur_clean.median(), color='#ef4444', linestyle='--', label=f'Median: {dur_clean.median():.0f} min')
axes[0,0].set_title('Incident Duration Distribution (0–24h)', color=YELLOW)
axes[0,0].set_xlabel('Minutes')
axes[0,0].legend(labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

# Resolution time
axes[0,1].hist(res_clean, bins=40, color='#22d3ee', alpha=0.8, edgecolor='none')
axes[0,1].axvline(res_clean.median(), color='#ef4444', linestyle='--', label=f'Median: {res_clean.median():.0f} min')
axes[0,1].set_title(f'Resolution Time Distribution (n={len(res_clean)})', color=YELLOW)
axes[0,1].set_xlabel('Minutes')
axes[0,1].legend(labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

# Avg duration by cause (valid only)
avg_dur = (df[df['duration_min'].between(0, 1440)]
           .groupby('event_cause')['duration_min']
           .median()
           .sort_values(ascending=False))
axes[1,0].barh(avg_dur.index, avg_dur.values, color=PALETTE)
axes[1,0].set_title('Median Duration by Cause (minutes)', color=YELLOW)
axes[1,0].invert_yaxis()

# High vs Low priority duration comparison
dur_pri = df[df['duration_min'].between(0,1440)].groupby('priority')['duration_min']
for i, (pri, grp) in enumerate(dur_pri):
    axes[1,1].hist(grp, bins=40, alpha=0.6, label=pri, color=PALETTE[i])
axes[1,1].set_title('Duration: High vs Low Priority', color=YELLOW)
axes[1,1].set_xlabel('Minutes')
axes[1,1].legend(labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.show()

print(f'\n🔍 INSIGHT: Median incident duration = {dur_clean.median():.0f} min')
print(f'   Median resolution time = {res_clean.median():.0f} min (only {len(res_clean)} incidents have this data)')
print(f'   → Most incidents clear in under 6 hours; water_logging and construction are outliers')

## 7. Road Closure & Severity Patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Road Closure & Priority Patterns', fontsize=14, color=YELLOW, fontweight='bold')

# Road closure rate by cause
closure_rate = (df.groupby('event_cause')['requires_road_closure']
                .mean() * 100
                .sort_values(ascending=False))
closure_rate = df.groupby('event_cause')['requires_road_closure'].mean() * 100
closure_rate = closure_rate.sort_values(ascending=False)
colors_rc = ['#ef4444' if v > 20 else '#f97316' if v > 10 else '#22d3ee' for v in closure_rate.values]
axes[0].barh(closure_rate.index, closure_rate.values, color=colors_rc)
axes[0].set_title('Road Closure Rate by Cause (%)', color=YELLOW)
axes[0].invert_yaxis()
axes[0].axvline(8.3, color='#8b949e', linestyle='--', label='Overall avg')
axes[0].legend(labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

# Priority by corridor (top 10)
top10_corridors = df['corridor'].value_counts().head(10).index
corr_pri = pd.crosstab(
    df[df['corridor'].isin(top10_corridors)]['corridor'],
    df[df['corridor'].isin(top10_corridors)]['priority'],
    normalize='index'
) * 100
corr_pri.plot(kind='barh', stacked=True, ax=axes[1], color=['#ef4444','#10b981'])
axes[1].set_title('Priority Split by Corridor (%)', color=YELLOW)
axes[1].invert_yaxis()
axes[1].legend(title='Priority', labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')
axes[1].set_xlabel('% of incidents')

# Night vs high priority
night_pri = pd.crosstab(df['is_night'], df['priority'], normalize='index') * 100
night_pri.index = ['Day (06–20)', 'Night (20–06)']
night_pri.plot(kind='bar', ax=axes[2], color=['#ef4444','#10b981'], rot=0)
axes[2].set_title('Priority Split: Day vs Night', color=YELLOW)
axes[2].legend(title='Priority', labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

vip_closure = df[df['event_cause']=='vip_movement']['requires_road_closure'].mean()*100
pub_closure = df[df['event_cause']=='public_event']['requires_road_closure'].mean()*100
print(f'\n🔍 INSIGHT: VIP movement = {vip_closure:.0f}% road closure rate — highest of all causes')
print(f'   Public events = {pub_closure:.0f}% — also very high')
print(f'   → These causes should auto-trigger ROAD CLOSURE flag in DRISHTI')

## 8. Vehicle Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Vehicle Type Patterns', fontsize=14, color=YELLOW, fontweight='bold')

# Vehicle type frequency
veh = df['veh_type'].value_counts().dropna()
axes[0].pie(veh.values, labels=veh.index, colors=PALETTE,
            autopct='%1.1f%%', textprops={'color':'#c9d1d9', 'fontsize':9},
            wedgeprops={'edgecolor':'#0d1117','linewidth':1.5})
axes[0].set_title('Vehicle Type Distribution\n(3,286 NaN = non-vehicle incidents)', color=YELLOW)

# Avg priority by vehicle type
veh_high = (df[df['veh_type'].notna()]
            .assign(is_high=lambda x: (x['priority'] == 'High').astype(int))
            .groupby('veh_type')['is_high']
            .mean() * 100
            .sort_values(ascending=False))
axes[1].bar(veh_high.index, veh_high.values, color=PALETTE)
axes[1].set_title('% High Priority by Vehicle Type', color=YELLOW)
axes[1].set_xticklabels(veh_high.index, rotation=30, ha='right')
axes[1].axhline(df[df['priority'].notna()].assign(h=lambda x: (x['priority']=='High').astype(int))['h'].mean()*100,
                color='#8b949e', linestyle='--', label='Overall avg')
axes[1].legend(labelcolor='#c9d1d9', facecolor='#161b22', edgecolor='#30363d')

plt.tight_layout()
plt.show()

print('\n🔍 INSIGHT: BMTC buses are the most common vehicle involved after heavy vehicles')
print('   → Public transport disruptions need fastest response — route impact is highest')

## 9. Corridor × Hour Heatmap (When Do Hotspot Corridors Peak?)

In [ ]:
top_corridors_list = [
    'Mysore Road','Bellary Road 1','Tumkur Road','Bellary Road 2',
    'Hosur Road','ORR North 1','Old Madras Road','ORR East 1','Magadi Road','ORR North 2'
]
heat_data = (df[df['corridor'].isin(top_corridors_list)]
             .groupby(['corridor','hour'])
             .size()
             .unstack(fill_value=0))
heat_data = heat_data.reindex(columns=range(0,24), fill_value=0)

fig, ax = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor('#0d1117')
sns.heatmap(heat_data, ax=ax, cmap='YlOrRd', linewidths=0.3, linecolor='#0d1117',
            annot=True, fmt='d', annot_kws={'size':7}, cbar_kws={'label':'# incidents'})
ax.set_title('Incident Count: Top Corridors × Hour of Day', color=YELLOW, fontsize=13, pad=12)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('')
ax.tick_params(colors='#c9d1d9')
plt.tight_layout()
plt.show()

print('\n🔍 INSIGHT: Mysore Road and Bellary Road peak at 5–7 AM and 9–10 PM')
print('   → These are highway corridors — DRISHTI patrol schedule should match this pattern')

## 10. Key Actionable Insights Summary

In [ ]:
insights = [
    ('Vehicle Breakdown = 60% of incidents',
     'Add a dedicated fast-track complaint type in DRISHTI form with predefined fields'),
    ('73% of incidents happen at night (20:00–06:00)',
     'DRISHTI patrol scheduling should weight night shifts; alert thresholds lower at night'),
    ('Mysore Rd · Bellary Rd 1 · Tumkur Rd = top 3 corridors',
     'Pre-fill these in DRISHTI drop-downs; these corridors need dedicated patrol slots'),
    ('VIP movement & public events have 80%+ road closure rate',
     'Auto-set requires_road_closure=True when these causes are detected'),
    ('58% of incidents have no zone tag',
     'DRISHTI MapmyIndia geocoder fills this — improving data completeness over time'),
    ('BMTC buses are most common vehicle in incidents',
     'Route disruption alerts: notify BMTC operations automatically for bus incidents'),
    ('Thursday & Tuesday have peak incidents; Sunday has lowest',
     'Dynamic officer scheduling: more deployments mid-week, reduced on Sunday'),
    ('Median resolution time = 59 min (small sample)',
     'DRISHTI can benchmark officer performance against this baseline'),
]

fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('#0d1117')
ax.axis('off')

ax.text(0.5, 1.0, 'DRISHTI — EDA Action Items', transform=ax.transAxes,
        ha='center', va='top', fontsize=14, color=YELLOW, fontweight='bold')

y = 0.92
for i, (finding, action) in enumerate(insights):
    ax.text(0.02, y, f'  {finding}', transform=ax.transAxes,
            ha='left', va='top', fontsize=10, color='#ef4444', fontweight='bold')
    ax.text(0.02, y - 0.048, f'    → {action}', transform=ax.transAxes,
            ha='left', va='top', fontsize=9, color='#c9d1d9')
    y -= 0.11

plt.tight_layout()
plt.show()

## 11. Export Hotspot Reference Data (feeds DRISHTI map & ML)

In [ ]:
import json, os

# 1. Corridor risk scores
corridor_stats = (df[df['corridor'] != 'Non-corridor']
    .groupby('corridor')
    .agg(
        total_incidents    = ('id', 'count'),
        high_priority_pct  = ('priority', lambda x: (x == 'High').mean() * 100),
        road_closure_pct   = ('requires_road_closure', 'mean'),
        night_pct          = ('is_night', 'mean'),
    )
    .round(2)
    .reset_index()
    .sort_values('total_incidents', ascending=False)
)
corridor_stats.to_csv('../backend/ml/corridor_risk_scores.csv', index=False)
print('Saved corridor_risk_scores.csv')

# 2. Junction hotspots with GPS
junction_gps = (df.dropna(subset=['junction','latitude','longitude'])
    .groupby('junction')
    .agg(
        count     = ('id', 'count'),
        lat       = ('latitude',  'median'),
        lng       = ('longitude', 'median'),
        high_pct  = ('priority', lambda x: (x=='High').mean()),
    )
    .reset_index()
    .sort_values('count', ascending=False)
    .head(50)
)
junction_gps.to_csv('../backend/ml/junction_hotspots.csv', index=False)
print('Saved junction_hotspots.csv')

# 3. Hourly incident rates (for patrol scheduling)
hourly_stats = (df.groupby('hour')
    .agg(
        incidents    = ('id', 'count'),
        high_pct     = ('priority', lambda x: (x=='High').mean()),
        closure_pct  = ('requires_road_closure', 'mean'),
    )
    .reset_index()
    .round(3)
)
hourly_stats.to_csv('../backend/ml/hourly_incident_rates.csv', index=False)
print('Saved hourly_incident_rates.csv')

print(f'\nAll 3 reference files exported to backend/ml/')
print('These feed: corridor risk in ML model, junction pins on DRISHTI map, patrol scheduler')